# Q-A Evaluating metrics

Question Answering Language Models (LLMs) are typically evaluated using a variety of metrics to assess their accuracy and effectiveness in answering questions. Here are some of the common metrics used for evaluating QA LLMs:



In [ ]:
#!pip install langchain openai chromadb unstructured

In [59]:
# Import package

from PyPDF2 import PdfReader
import pandas as pd

import re
import warnings
from sklearn.metrics import f1_score
import nltk
from rouge import Rouge
from jiwer import wer
from evaluate import load

from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.docstore.document import Document
from langchain.prompts import PromptTemplate
from langchain.chains.question_answering import load_qa_chain
from langchain.llms import OpenAI
from langchain.document_loaders import TextLoader, DirectoryLoader
from langchain.chains.qa_with_sources import load_qa_with_sources_chain
from dotenv import load_dotenv, find_dotenv
import os

warnings.filterwarnings("ignore")

In [46]:
load_dotenv(find_dotenv("credentials.env"), override=True)

True

**Exact Match (EM)**

The Exact Match (EM) metric is a binary metric used to evaluate the performance of a question-answering system. It measures whether the generated answer by the LLM exactly matches the correct answer. If the generated answer is identical to the correct answer, then the EM score is 1, indicating a perfect match between the generated answer and the correct answer. Otherwise, if the generated answer is different from the correct answer, the EM score is 0, indicating that the LLM did not produce the correct answer.

In [47]:
def normalize_answer(text):
    # Lowercase and remove non-alphanumeric characters
    text = re.sub(r"\W", " ", text.lower())
    # Remove leading and trailing whitespace
    text = text.strip()
    # Collapse multiple whitespace into a single whitespace
    text = re.sub(r"\s+", " ", text)
    return text

def exact_match_score(prediction, ground_truth):
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

# Example usage
prediction = "I am going to school"
ground_truth = "I am going to college"
em_score = exact_match_score(prediction, ground_truth)
print(f"Exact Match score: {em_score}")

Exact Match score: 0


The EM metric is a simple and straightforward way to evaluate the performance of a question-answering system, and it is often used in conjunction with other metrics, such as the F1 score, to provide a more comprehensive evaluation of the system's performance. However, the EM metric has its limitations, as it is very strict and unforgiving, and it does not take into account partial matches or variations in the correct answer.
For example, if the correct answer is "I am going to college", and the generated answer is "I am going to school", the EM score would be 0, even though the generated answer is very close to the correct answer. Therefore, the EM metric should be used in conjunction with other metrics to provide a more comprehensive evaluation of the performance of a question-answering system.

**F1-Score**

The F1 score is a commonly used metric for evaluating the performance of question-answering systems, particularly in cases where the answer is expected to be a span of text within a larger document. The F1 score is a harmonic mean of precision and recall, which are two other commonly used metrics in natural language processing.The F1 score is the harmonic mean of precision and recall, and it provides a single metric that balances both precision and recall. The F1 score ranges from 0 to 1, with a higher score indicating better performance. An F1 score of 1 means that the LLM generated the correct answer with 100% accuracy and completeness, while a score of 0 means that the LLM did not generate any correct answers.



In [48]:
def f1_score_metric(prediction, ground_truth):
    labels = list(set([prediction, ground_truth]))
    prediction_label = labels.index(prediction)
    ground_truth_label = labels.index(ground_truth)
    return f1_score([ground_truth_label], [prediction_label], average='weighted')

# Example usage
prediction = "I am going to school"
ground_truth = "I am going to college"
f1 = f1_score_metric(prediction, ground_truth)
print(f"F1 score: {f1}")

F1 score: 0.0


In question answering, the F1 score is often used in conjunction with other metrics, such as the Exact Match (EM) score, to provide a more comprehensive evaluation of the system's performance. The F1 score is particularly useful when the correct answer is expected to be a span of text within a larger document, as it takes into account the position and length of the generated answer in relation to the correct answer.

**BLEU (Bilingual Evaluation Understudy) Score**

BLEU (Bilingual Evaluation Understudy) Score is a metric used to evaluate the quality of machine-generated text against human-generated text. It is commonly used to evaluate the performance of machine translation systems, but it can also be used to evaluate other text generation tasks such as summarization, question answering, and text completion.

The BLEU score is calculated by first computing the precision of the machine-generated text for each n-gram size from 1 to N, where N is typically 4. Precision is the number of n-grams in the machine-generated text that also appear in the human-generated text, divided by the total number of n-grams in the machine-generated text. The precision values are then combined using a geometric mean to produce the final BLEU score.

One limitation of the BLEU score is that it does not take into account the fluency or coherence of the machine-generated text, only its similarity to the human-generated text in terms of n-gram overlap. For this reason, the BLEU score should be used in conjunction with other metrics such as perplexity and human evaluation to provide a more comprehensive evaluation of the quality of the machine-generated text.

In [49]:
def bleu_score_metric(prediction, ground_truth):
    reference = [ground_truth.split()]
    candidate = prediction.split()
    return nltk.translate.bleu_score.sentence_bleu(reference, candidate)

# Example usage
prediction = "I am going to school"
ground_truth = "I am going to college"
bleu = bleu_score_metric(prediction, ground_truth)
print(f"BLEU score: {bleu}")

BLEU score: 0.668740304976422


The score ranges from 0 to 1, with higher scores indicating better performance. However, it is important to note that the BLEU score has its limitations, and should be used in combination with other metrics and human evaluation to fully assess the quality and usefulness of machine-generated text. When interpreting BLEU scores, extremely low scores (e.g., on the order of 1e-154 or 1e-231) suggest that the machine-generated text is very dissimilar to the reference text, while scores closer to 1 suggest that the machine-generated text is more similar to the reference text.

**ROUGE (Recall-Oriented Understudy for Gisting Evaluation) Score**

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) Score is a metric used to evaluate the quality of machine-generated text against human-generated text. It is commonly used to evaluate the performance of text summarization systems, but it can also be used to evaluate other text generation tasks such as machine translation and question answering.

ROUGE score measures the overlap between the machine-generated summary and the human-generated summary using n-gram co-occurrence statistics. It is based on the recall of n-gram overlapping between machine-generated text and the reference (human-generated text). The score ranges from 0 to 1, with 1 indicating a perfect match between the two texts. The higher the ROUGE score, the better the machine-generated text is considered to be.



In [50]:
def rouge_score_metric(prediction, ground_truth):
    rouge = Rouge()
    scores = rouge.get_scores(prediction, ground_truth)
    return scores[0]["rouge-1"]["f"]

# Example usage
prediction = "I am going to school"
ground_truth = "I am going to college"
rouge = rouge_score_metric(prediction, ground_truth)
print(f"ROUGE score: {rouge}")

ROUGE score: 0.7999999950000002


In above case, the ROUGE score is 0.7999999950000002, which is a relatively high score and suggests that there is a good amount of overlap between the model-generated text and reference text. One limitation of the ROUGE score is that it only evaluates the overlap between the machine-generated text and the reference, without considering other important factors such as fluency, coherence, and content preservation. As a result, the ROUGE score should be used in conjunction with other metrics such as BLEU score, perplexity, and human evaluation to provide a more comprehensive evaluation of the quality of the machine-generated text.


**WER (Word Error Rate)**

WER (Word Error Rate) is a metric used for evaluating the performance of speech recognition systems or OCR (Optical Character Recognition) systems. It measures the percentage of words in the recognized text that differ from the words in the reference text. WER is calculated by dividing the total number of word errors (insertions, deletions, and substitutions) by the total number of words in the reference text. 

In [51]:
# Example usage
prediction = "I am going to school"
ground_truth = "I am going to college"
wer = wer(prediction, ground_truth)
print(f"WER score: {wer}")

WER score: 0.2


A WER (Word Error Rate) score of 0.2 means that 20% of the words in the recognized text are different from the words in the reference text. In other words, there is an error rate of 20%. 

# Evaluation for Generative QA model

### [BERTScore](https://github.com/Tiiiger/bert_score)

BERTScore is a metric that evaluates the quality of generated text by comparing it to reference text. BERTScore uses the pre-trained BERT (Bidirectional Encoder Representations from Transformers) model to encode both the generated text and reference text. It then calculates the similarity between the two encodings by computing the cosine similarity between them. BERTScore has been shown to outperform other embedding-based metrics like ROUGE and BLEU in terms of correlation with human judgments of text quality.

In [52]:
#Result [WIP]

from evaluate import load
bertscore = load("bertscore")
predictions = ["hello world", "I am going to school"]
references = ["hello world", "I am going to college"]
results = bertscore.compute(predictions=predictions, references=references, model_type="distilbert-base-uncased")
print(results)


{'precision': [1.0000001192092896, 0.9577858448028564], 'recall': [1.0000001192092896, 0.9577858448028564], 'f1': [1.0000001192092896, 0.9577858448028564], 'hashcode': 'distilbert-base-uncased_L5_no-idf_version=0.3.12(hug_trans=4.26.1)'}


In this case, the output shows that the precision, recall, and F1 score values for the first phrase "hello world" are all 1.0, indicating that the generated text is identical to the reference text. However, for the second phrase "I am going to school", the precision, recall, and F1 score values are all slightly lower, at around 0.96. This indicates that the generated text is similar to the reference text, but not identical.

---

In the next case, we will calculate these evaluation metric, on squad dataset.

# Generating Answers 

Here, in order to generate answers. We explore the langchain framework, also described in this [notebook](https://github.com/Shreyanand/foundation-models-for-documentation/blob/langchain-openai/notebooks/langchain-openai.ipynb). We will be using OpenAI api as a choice of LLM.

In [54]:
loader = TextLoader('../data/external/rosaworkshop/14-faq.txt')
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
embeddings = OpenAIEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings)

Running Chroma using direct local API.
Using DuckDB in-memory for database. Data will be transient.


In [55]:
query = "What is Red Hat OpenShift Service on AWS (ROSA)?"
docs = docsearch.similarity_search(query)
print(docs[0].page_content)

Red Hat OpenShift Service on AWS - FAQ

General

What is Red Hat OpenShift Service on AWS (ROSA)?

Red Hat Openshift Service on AWS (ROSA) is a fully-managed turnkey application platform that allows you to focus on what matters most, delivering value to your customers by building and deploying applications. Red Hat SRE experts manage the underlying platform so you don’t have to worry about the complexity of infrastructure management.

Where can I go to get more information/details?

-   ROSA Webpage
-   ROSA Workshop
-   ROSA Documentation

What are the benefits of Red Hat OpenShift Service on AWS (Key Features)?


**Example query and answer**

In [57]:
query = 'Where is the purpose of this document?'
chain.run(input_documents=docs, question=query)

' This document is a FAQ about Red Hat OpenShift Service on AWS.'

# Validation Dataset

We create a validation dataset by extracting question and their respective answers from the FAQ text provided. 

In [60]:
def pdf_to_squad(pdf_file):
    # Open the PDF file
    pdf = PdfReader(pdf_file)

    # Initialize the QA squad dataset list
    squad_dataset = []

    # Get the number of pages in the PDF
    num_pages = len(pdf.pages)

    # Loop through each page in the PDF
    for i in range(num_pages):
        # Get the text from each page
        page = pdf.pages[i].extract_text()
        
        # Split the text into sentences
        sentences = page.split('.')
        
        # Loop through each sentence to create a QA pair
        for sentence in sentences:
            try:
                # Split the sentence into question and answer
                question, answer = sentence.split('?')
                
                # Add the QA pair to the squad dataset list
                squad_dataset.append({
                    'question': question.strip(),
                    'answer': answer.strip()
                })
            except ValueError:
                # If the sentence does not contain a question and answer, skip it
                continue
    
    return squad_dataset

In [61]:
pdf_file = 'RH-1-productfaq.pdf'
squad_dataset = pdf_to_squad('RH-1-productfaq.pdf')

squad_dataset = pd.DataFrame(squad_dataset)

Exiting: Cleaning up .chroma directory


In [62]:
squad_dataset

,question,answer
0,General questions\nWhat does Red Hat OpenShift...,Each Red Hat OpenShift Service on AWS cluster ...
1,How is Red Hat OpenShift Service on AWS differ...,Red Hat OpenShift Service on AWS delivers a tu...
2,How Is Red Hat OpenShift Service on AWS differ...,Red Hat OpenShift Service on AWS is a fully ma...
3,What are the differences between Red Hat OpenS...,Red Hat OpenShift
4,Will Red Hat OpenShift Service on AWS integrat...,Yes
...,...,...
70,Can I define a custom domain and certificate f...,Yes
71,How are the ROSA domain certificates managed,Red Hat infrastructure (Hive) manages certific...
72,com)\nWhat features are upcoming for ROSA,The current ROSA roadmap can be seen at: https...
73,ht/rosa-roadmap\nWhat kind of instances are su...,See AWS compute types in the service definitio...


Now using the Langchain OpenAI api, we will generate answers corresponding to the questions from the dataset above.

In [63]:
generated_answer = []
df =squad_dataset

In [64]:
# Loop through each row in the DataFrame and generate an answer
for index, row in df.iterrows():
    query = row['question']
    answer = chain.run(input_documents=docs, question=query)
    generated_answer.append(answer)

In [65]:
# Add the generated answers as a new column to the DataFrame
df['generated_answer'] = generated_answer

In [66]:
df

,question,answer,generated_answer
0,General questions\nWhat does Red Hat OpenShift...,Each Red Hat OpenShift Service on AWS cluster ...,Red Hat OpenShift Service on AWS (ROSA) inclu...
1,How is Red Hat OpenShift Service on AWS differ...,Red Hat OpenShift Service on AWS delivers a tu...,Red Hat OpenShift Service on AWS (ROSA) is a ...
2,How Is Red Hat OpenShift Service on AWS differ...,Red Hat OpenShift Service on AWS is a fully ma...,Red Hat OpenShift Service on AWS is a fully-m...
3,What are the differences between Red Hat OpenS...,Red Hat OpenShift,Red Hat OpenShift Service on AWS (ROSA) is a ...
4,Will Red Hat OpenShift Service on AWS integrat...,Yes,"Yes, Red Hat OpenShift Service on AWS integra..."
...,...,...,...
70,Can I define a custom domain and certificate f...,Yes,"Yes, you can define a custom domain and certi..."
71,How are the ROSA domain certificates managed,Red Hat infrastructure (Hive) manages certific...,ROSA domain certificates are managed by the R...
72,com)\nWhat features are upcoming for ROSA,The current ROSA roadmap can be seen at: https...,You can visit our ROSA roadmap to stay up to ...
73,ht/rosa-roadmap\nWhat kind of instances are su...,See AWS compute types in the service definitio...,Red Hat OpenShift Service on AWS is a managed...


As we see, we now have questions, answers, and generated answers. In the next case, we will be evaluating each metrics, and saving them as a column in the dataset. 

In [68]:
from evaluate import load
bertscore = load("bertscore")

def calculate_scores(row, model_type="distilbert-base-uncased"):
    question = row['question']
    answer = row['answer']
    generated_answer = row['generated_answer']
    
    # Exact match
    exact_match = exact_match_score(generated_answer, answer)
    
    # BLEU score
    bleu_score = bleu_score_metric(generated_answer, answer)
    
    # ROUGE score
    rouge_score = rouge_score_metric(generated_answer, answer)
    
    # WER score
    wer_score = wer(generated_answer, answer)
    
    # BERT score
    results = bertscore.compute(predictions=[generated_answer], references=[answer], model_type=model_type)
    bert_f1_score = results["f1"][0]
    
    return question, answer, generated_answer, exact_match, bleu_score, rouge_score, wer_score, bert_f1_score




In [69]:
new_dataset = df.apply(calculate_scores, axis=1, result_type='expand')
new_dataset.columns = ['question', 'answer', 'generated_answer', 'exact_match', 'BLEU_score', 'ROUGE_score', 'WER_Score', "BERT_F1_score"]

In [70]:
new_dataset

,question,answer,generated_answer,exact_match,BLEU_score,ROUGE_score,WER_Score,BERT_F1_score
0,General questions\nWhat does Red Hat OpenShift...,Each Red Hat OpenShift Service on AWS cluster ...,Red Hat OpenShift Service on AWS (ROSA) inclu...,0,1.216278e-01,0.275862,0.825000,0.817213
1,How is Red Hat OpenShift Service on AWS differ...,Red Hat OpenShift Service on AWS delivers a tu...,Red Hat OpenShift Service on AWS (ROSA) is a ...,0,7.675187e-02,0.298507,0.855072,0.811004
2,How Is Red Hat OpenShift Service on AWS differ...,Red Hat OpenShift Service on AWS is a fully ma...,Red Hat OpenShift Service on AWS is a fully-m...,0,9.479302e-02,0.302326,0.867347,0.843002
3,What are the differences between Red Hat OpenS...,Red Hat OpenShift,Red Hat OpenShift Service on AWS (ROSA) is a ...,0,6.710406e-79,0.083333,0.988636,0.724214
4,Will Red Hat OpenShift Service on AWS integrat...,Yes,"Yes, Red Hat OpenShift Service on AWS integra...",0,0.000000e+00,0.000000,1.000000,0.641446
...,...,...,...,...,...,...,...,...
70,Can I define a custom domain and certificate f...,Yes,"Yes, you can define a custom domain and certi...",0,0.000000e+00,0.000000,1.000000,0.667207
71,How are the ROSA domain certificates managed,Red Hat infrastructure (Hive) manages certific...,ROSA domain certificates are managed by the R...,0,4.124993e-155,0.133333,0.894737,0.774967
72,com)\nWhat features are upcoming for ROSA,The current ROSA roadmap can be seen at: https...,You can visit our ROSA roadmap to stay up to ...,0,3.295034e-155,0.142857,0.944444,0.740735
73,ht/rosa-roadmap\nWhat kind of instances are su...,See AWS compute types in the service definitio...,Red Hat OpenShift Service on AWS is a managed...,0,9.181749e-232,0.095238,0.967742,0.743827


# Conclusion

In this work, we generated answers using the OpenAI language model and evaluated their quality against the ground truth answers for a set of questions related to FAQ doc. We used several evaluation metrics such as exact match, BLEU score, ROUGE score, WER score, and BERT F1 score to compare the generated answers with the ground truth.